In [ ]:
# lightgbm 라이브러리(파이썬 패키지)를 임포트
# LightGBM은 Microsoft에서 개발한 GBM(Gradient Boosting Machine) 기반의 부스팅 알고리즘으로,
# XGBoost와 유사한 성능을 보이면서도 학습 속도가 훨씬 빠르고 메모리 사용량이 적은 것이 특징.
# - 리프 중심 트리 분할(Leaf-wise) 방식 사용 → 균형 트리 분할(Level-wise)보다 학습 효율 높음
# - 다만 데이터 양이 적을 경우(약 10,000건 이하) 과적합 발생 가능성이 있음
import lightgbm

# 현재 설치된 lightgbm 버전 정보를 출력
# (버전에 따라 사용 가능한 파라미터 및 API가 일부 다를 수 있으므로 확인 필요)
print(lightgbm.__version__)

### LightGBM 적용 – 위스콘신 Breast Cancer Prediction

In [ ]:
# LightGBM의 파이썬 패키지인 lightgbm에서 LGBMClassifier 임포트
# LGBMClassifier는 사이킷런 래퍼 클래스로, fit/predict 등 사이킷런과 동일한 인터페이스를 제공
from lightgbm import LGBMClassifier

# 데이터 처리/분석에 사용하는 pandas, numpy 임포트
import pandas as pd
import numpy as np
# 사이킷런에서 제공하는 위스콘신 유방암 데이터셋 로더 임포트
from sklearn.datasets import load_breast_cancer
# 학습/테스트 데이터 분리 함수 임포트
from sklearn.model_selection import train_test_split

# 위스콘신 유방암 데이터셋 로드 (Bunch 객체 형태 반환)
dataset = load_breast_cancer()

# 피처 데이터를 DataFrame으로 변환 (컬럼명은 dataset.feature_names로 지정)
cancer_df = pd.DataFrame(data=dataset.data, columns=dataset.feature_names)
# 마지막에 'target' 컬럼 추가 (0: 악성, 1: 양성)
cancer_df['target']= dataset.target
# 피처 데이터: 마지막 컬럼('target')을 제외한 모든 컬럼
X_features = cancer_df.iloc[:, :-1]
# 레이블 데이터: 마지막 컬럼('target')
y_label = cancer_df.iloc[:, -1]

# 전체 데이터 중 80%는 학습용 데이터, 20%는 테스트용 데이터 추출
# random_state=156: 동일한 분할 결과 재현을 위한 시드 고정값
X_train, X_test, y_train, y_test=train_test_split(X_features, y_label, test_size=0.2, random_state=156 )

# 위에서 만든 X_train, y_train을 다시 쪼개서 90%는 학습과 10%는 검증용 데이터로 분리
# 검증(validation) 데이터는 조기 중단(early stopping) 평가에 사용
X_tr, X_val, y_tr, y_val= train_test_split(X_train, y_train, test_size=0.1, random_state=156 )

# 앞서 XGBoost와 동일하게 n_estimators는 400 설정.
# n_estimators=400: 생성할 부스팅 트리의 최대 개수
# learning_rate=0.05: 학습률. 각 트리가 결과에 기여하는 비율 (작을수록 학습 안정성 ↑, 속도 ↓)
lgbm_wrapper = LGBMClassifier(n_estimators=400, learning_rate=0.05)

# LightGBM도 XGBoost와 동일하게 조기 중단 수행 가능.
# evals 리스트에 (학습용, 검증용) 두 데이터셋을 함께 넣어 학습 중 성능을 모니터링
evals = [(X_tr, y_tr), (X_val, y_val)]
# fit() 메서드로 학습 수행
# early_stopping_rounds=50: 검증 데이터의 평가지표가 50회 연속 개선되지 않으면 학습 조기 종료
# eval_metric='logloss': 검증 평가에 사용할 지표 (이진 분류에서 일반적으로 사용)
# eval_set=evals: 학습 중 평가에 사용할 데이터셋 목록
# verbose=True: 학습 진행 상황(라운드별 logloss)을 콘솔에 출력
lgbm_wrapper.fit(X_tr, y_tr, early_stopping_rounds=50, eval_metric="logloss", eval_set=evals, verbose=True)
# 학습된 모델로 테스트 데이터에 대한 클래스 예측 (0 또는 1)
preds = lgbm_wrapper.predict(X_test)
# 클래스별 확률 예측 후, 양성(1) 클래스 확률만 슬라이싱 (ROC-AUC 계산용)
pred_proba = lgbm_wrapper.predict_proba(X_test)[:, 1]

In [ ]:
# 분류 성능 평가에 사용할 사이킷런 메트릭 함수들 임포트
from sklearn.metrics import confusion_matrix, accuracy_score   # 오차행렬, 정확도
from sklearn.metrics import precision_score, recall_score      # 정밀도, 재현율
from sklearn.metrics import f1_score, roc_auc_score            # F1 스코어, ROC-AUC

# 분류 모델 성능 평가 결과를 한 번에 출력해주는 사용자 정의 함수
# y_test: 실제 정답 레이블
# pred: 예측 클래스 (0/1)
# pred_proba: 양성 클래스 예측 확률 (ROC-AUC 계산에 필요)
def get_clf_eval(y_test, pred=None, pred_proba=None):
    # 오차행렬: [[TN, FP], [FN, TP]] 형태로 반환
    confusion = confusion_matrix( y_test, pred)
    # 정확도: 전체 중 올바르게 예측한 비율
    accuracy = accuracy_score(y_test , pred)
    # 정밀도: 양성으로 예측한 것 중 실제 양성 비율 (TP / (TP+FP))
    precision = precision_score(y_test , pred)
    # 재현율: 실제 양성 중 양성으로 예측한 비율 (TP / (TP+FN))
    recall = recall_score(y_test , pred)
    # F1 스코어: 정밀도와 재현율의 조화 평균 (불균형 데이터에 적합한 지표)
    f1 = f1_score(y_test,pred)
    # ROC-AUC 추가 
    # ROC-AUC: 임계값 변화에 따른 분류 성능을 종합적으로 나타내는 지표 (1에 가까울수록 우수)
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    # ROC-AUC print 추가
    # 평가 지표 5종을 소수점 4자리까지 한 줄로 출력
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [ ]:
# 위에서 정의한 get_clf_eval() 함수에 실제 정답(y_test), 예측 클래스(preds),
# 양성 클래스 예측 확률(pred_proba)을 전달해 LightGBM 모델의 분류 성능 평가 결과를 출력
get_clf_eval(y_test, preds, pred_proba)

In [ ]:
# plot_importance( )를 이용하여 feature 중요도 시각화
# LightGBM도 XGBoost와 마찬가지로 plot_importance 함수를 통해 피처 중요도 시각화 가능
from lightgbm import plot_importance
import matplotlib.pyplot as plt
# 주피터 노트북 내에서 그래프를 인라인으로 출력하기 위한 매직 커맨드
%matplotlib inline

# 가로 10, 세로 12 크기의 Figure(전체 그림)와 Axes(서브 플롯) 생성
fig, ax = plt.subplots(figsize=(10, 12))
# 학습된 LightGBM 모델(lgbm_wrapper)의 피처 중요도(Feature Importance)를 막대 그래프로 시각화
# (기본적으로 split 기준: 각 피처가 트리 분할에 사용된 횟수 기준으로 정렬)
plot_importance(lgbm_wrapper, ax=ax)
# 시각화한 피처 중요도 그래프를 TIFF 이미지 파일로 저장 (해상도 300 dpi, 여백 자동 조정)
plt.savefig('lightgbm_feature_importance.tif', format='tif', dpi=300, bbox_inches='tight')